# Customer Support Call Analysis

## Tasks

1. Determine whether the audio is compatible for future speech recognition modeling.
2. Convert `sample_customer_call.wav` into text and store the result in `transcribed_text`.
3. Find the audio frame rate and number of channels and store them in:
   - `frame_rate`
   - `number_channels`
4. Perform sentiment analysis on `customer_call_transcriptions.csv` using VADER:
   - Compound score `>= 0.05` → Positive
   - Compound score `<= -0.05` → Negative
   - Otherwise → Neutral
5. Count the number of **true positive** sentiment predictions and store it in `true_positive`.
6. Perform named entity recognition across all transcriptions and find the most frequently occurring named entity. Store it in `most_freq_ent`.
7. Find the transcription most similar to `"wrong package delivery"` and store it in `most_similar_text`.



## Installing Required Libraries and spaCy English Model

In [1]:
#!pip install SpeechRecognition
#!pip install pydub
#!pip install spacy
#!python3 -m spacy download en_core_web_sm

In [2]:
# Import required libraries
import pandas as pd

import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

import speech_recognition as sr
from pydub import AudioSegment

import spacy

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\FAUZAN\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
c:\PROJECTS\Analyse customer support callas\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


# Task 1 - Speech to Text: convert the sample audio call, sample_customer_call.wav, to text and store the result in transcribed_text


In [ ]:


# Define a recognizer object
recognizer = sr.Recognizer()

# Convert the audio file to audio data
transcribe_audio_file = sr.AudioFile(r"C:\PROJECTS\Analyse customer support callas\sample_customer_call.wav")
with transcribe_audio_file as source:
    transcribe_audio = recognizer.record(source)

# Convert the audio data to text
transcribed_text = recognizer.recognize_google(transcribe_audio)

# Review trascribed text
print("Transcribed text: ", transcribed_text)



Transcribed text:  hello I am experiencing an issue with your product I like to speak to someone a better replacement


# Task 1 - Speech to Text: store few statistics of the audio file such as number of channels, sample width and frame rate


In [ ]:
    
# Review number of channels and frame rate of the audio file
audio_segment = AudioSegment.from_file(r"C:\PROJECTS\Analyse customer support callas\sample_customer_call.wav")
number_channels = audio_segment.channels
frame_rate = audio_segment.frame_rate

print("Number of channels: ", number_channels)
print("Frame rate: ", frame_rate)


Number of channels:  1
Frame rate:  44100


# Task 2 - Sentiment Analysis: use vader module from nltk library to determine the sentiment of each text of the customer_call_transcriptions.csv file and store them at a new sentiment_label column using compound score


In [ ]:

# Import customer call transcriptions data
df = pd.read_csv(r"C:\PROJECTS\Analyse customer support callas\customer_call_transcriptions.csv")

sid = SentimentIntensityAnalyzer()

# Analyze sentiment by evaluating compound score generated by Vader SentimentIntensityAnalyzer
def find_sentiment(text):
    scores = sid.polarity_scores(text)
    compound_score = scores['compound']

    if compound_score >= 0.05:
        return 'positive'
    elif compound_score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['sentiment_predicted'] = df.apply(lambda row: find_sentiment(row["text"]), axis = 1)


#  Sentiment Analysis: calculate number of texts with positive label that are correctly labeled as positive


In [ ]:
true_positive = len(df.loc[(df['sentiment_predicted'] == df['sentiment_label']) &
                (df['sentiment_label'] == 'positive')])

print("True positives: ", true_positive)


True positives:  2


# Task 3 - Named Entity Recognition: find named entities for each text in the df object and store entities in a named_entities column


In [ ]:

# Load spaCy small English Language model
nlp = spacy.load("en_core_web_sm")

# NER using spaCy
def extract_entities(text):
    doc = nlp(text)
    entities = [ent.text for ent in doc.ents]
    return entities

# Apply NER to the entire text column
df['named_entities'] = df['text'].apply(extract_entities)

# Flatten the list of named entities
all_entities = [ent for entities in df['named_entities'] for ent in entities]

# Create a DataFrame with the counts
entities_df = pd.DataFrame(all_entities, columns=['entity'])
entities_counts = entities_df['entity'].value_counts().reset_index()
entities_counts.columns = ['entity', 'count']

# Extract most frequent named entity
most_freq_ent = entities_counts["entity"].iloc[0]
print("Most frequent entity: ", most_freq_ent)


Most frequent entity:  yesterday


# Task 4 - Find most similar text: find the list of customer calls that complained about "wrong package delivery" by finding similarity score of each text to the "wrong package delivery" string using spaCy small English Language model


In [ ]:

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Process the text column
df['processed_text'] = df['text'].apply(lambda text: nlp(text))

# Input query
input_query = "wrong package delivery"
processed_query = nlp(input_query)

# Calculate similarity scores and sort dataframe with respect to similarity scores
df['similarity'] = df['processed_text'].apply(lambda text: processed_query.similarity(text))
df = df.sort_values(by='similarity', ascending=False)

# Find the most similar text
most_similar_text = df["text"].iloc[0]
print("Most similar text: ", most_similar_text)

Most similar text:  wrong package delivered


C:\Users\FAUZAN\AppData\Local\Temp\ipykernel_304\674144794.py:14: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  df['similarity'] = df['processed_text'].apply(lambda text: processed_query.similarity(text))
